In [ ]:
!pip install -q transformers peft accelerate sentencepiece scikit-learn

## Imports and Global Config

In [ ]:
import json
import math
import os
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from peft import LoraConfig, TaskType, get_peft_model
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float32

ROOT = Path("/content")

TRAIN_BUNDLE_PATH = ROOT / "train.bundle.json"
VAL_BUNDLE_PATH = ROOT / "val.bundle.json"
TEST_BUNDLE_PATH = ROOT / "test.bundle.json"

MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
OUTPUT_DIR = Path("./qwen3-episodic-lora")

NUM_HARD_NEGATIVES = 4
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 2
NUM_EPOCHS = 8
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
TEMPERATURE = 0.05
MAX_LENGTH = 512
QUERY_FORMAT = "raw"  # "instruction" follows Qwen3; "raw" matches the current benchmark CLI.
INSTRUCTION = (
    "Given a retrieval query, retrieve the exact prior agent episode that matches "
    "all relevant constraints, entities, and slot values."
)

print("DEVICE:", DEVICE)
print("DTYPE:", DTYPE)
print("TRAIN_BUNDLE_PATH:", TRAIN_BUNDLE_PATH)
print("VAL_BUNDLE_PATH:", VAL_BUNDLE_PATH)
print("TEST_BUNDLE_PATH:", TEST_BUNDLE_PATH)

DEVICE: cuda
DTYPE: torch.bfloat16
TRAIN_BUNDLE_PATH: /content/train.bundle.json
VAL_BUNDLE_PATH: /content/val.bundle.json
TEST_BUNDLE_PATH: /content/test.bundle.json


## Load Bundles

In [ ]:
def load_bundle(path: Path) -> dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


train_bundle = load_bundle(TRAIN_BUNDLE_PATH)
val_bundle = load_bundle(VAL_BUNDLE_PATH)
test_bundle = load_bundle(TEST_BUNDLE_PATH)

print("train episodes:", len(train_bundle["episodes"]))
print("train queries:", len(train_bundle["queries"]))
print("val episodes:", len(val_bundle["episodes"]))
print("val queries:", len(val_bundle["queries"]))
print("test episodes:", len(test_bundle["episodes"]))
print("test queries:", len(test_bundle["queries"]))

train episodes: 104
train queries: 104
val episodes: 20
val queries: 20
test episodes: 56
test queries: 56


## Episode Text Rendering

In [ ]:
def render_entities(entities: list[dict[str, Any]]) -> str:
    parts = []
    for entity in entities:
        entity_name = str(entity.get("name", "")).strip()
        if not entity_name:
            continue
        entity_type = str(entity.get("type", "")).strip()
        if entity_type:
            parts.append(f"{entity_type}={entity_name}")
        else:
            parts.append(entity_name)
    return "\n".join(parts)


def render_steps(steps: list[dict[str, Any]]) -> str:
    parts = []
    for step in steps:
        tool_name = str(step.get("tool_name", "")).strip()
        if tool_name:
            parts.append(tool_name)
    return " -> ".join(parts)


def build_episode_text(episode: dict[str, Any]) -> str:
    sections = []

    what_parts = []
    scenario = str(episode.get("scenario", "")).strip()
    intent = str(episode.get("intent", "")).strip()
    summary = str(episode.get("summary", "")).strip()
    if scenario:
        what_parts.append(f"scenario={scenario}")
    if intent:
        what_parts.append(f"intent={intent}")
    if summary:
        what_parts.append(f"summary={summary}")
    if what_parts:
        sections.append("what:\n" + "\n".join(what_parts))

    goal = str(episode.get("goal", "")).strip()
    if goal:
        sections.append("goal:\n" + goal)

    entities = render_entities(episode.get("entities", []))
    if entities:
        sections.append("task_content:\n" + entities)

    procedure = render_steps(episode.get("steps", []))
    if procedure:
        sections.append("procedure:\n" + procedure)

    return "\n\n".join(sections)


def build_query_text(
    query: str,
    instruction: str = INSTRUCTION,
    query_format: str = QUERY_FORMAT,
) -> str:
    query = query.strip()
    if query_format == "raw":
        return query
    if query_format == "instruction":
        return f"{instruction}\nQuery: {query}"
    raise ValueError(f"unsupported query_format: {query_format}")

## Build Train/Val Lookup Tables

In [ ]:
def index_episodes(bundle: dict[str, Any]) -> dict[str, dict[str, Any]]:
    return {episode["episode_id"]: episode for episode in bundle["episodes"]}


train_episode_by_id = index_episodes(train_bundle)
val_episode_by_id = index_episodes(val_bundle)


def group_episodes(bundle: dict[str, Any]) -> dict[tuple[str, str], list[dict[str, Any]]]:
    grouped: dict[tuple[str, str], list[dict[str, Any]]] = {}
    for episode in bundle["episodes"]:
        key = (episode["scenario"], episode["intent"])
        grouped.setdefault(key, []).append(episode)
    return grouped


train_grouped = group_episodes(train_bundle)

## Hard Negative Sampler

In [ ]:
def sample_hard_negatives(
    query_item: dict[str, Any],
    episode_by_id: dict[str, dict[str, Any]],
    grouped: dict[tuple[str, str], list[dict[str, Any]]],
    all_episodes: list[dict[str, Any]],
    num_negatives: int,
) -> list[dict[str, Any]]:
    target_id = query_item["target_episode_ids"][0]
    target_scenario = query_item["target_scenario"]
    target_intent = query_item["target_intent"]

    same_group = [
        ep
        for ep in grouped.get((target_scenario, target_intent), [])
        if ep["episode_id"] != target_id
    ]

    same_scenario_other_intent = [
        ep
        for ep in all_episodes
        if ep["episode_id"] != target_id
        and ep["scenario"] == target_scenario
        and ep["intent"] != target_intent
    ]

    global_pool = [
        ep for ep in all_episodes if ep["episode_id"] != target_id
    ]

    random.shuffle(same_group)
    random.shuffle(same_scenario_other_intent)
    random.shuffle(global_pool)

    chosen: list[dict[str, Any]] = []
    seen_ids = set()

    for pool in (same_group, same_scenario_other_intent, global_pool):
        for ep in pool:
            if ep["episode_id"] in seen_ids:
                continue
            chosen.append(ep)
            seen_ids.add(ep["episode_id"])
            if len(chosen) == num_negatives:
                return chosen

    if len(chosen) < num_negatives:
        raise ValueError(
            f"not enough negative episodes for target {target_id}: "
            f"wanted {num_negatives}, got {len(chosen)}"
        )

    return chosen

Dataset

In [ ]:
class EpisodicContrastiveDataset(Dataset):
    def __init__(
        self,
        bundle: dict[str, Any],
        episode_by_id: dict[str, dict[str, Any]],
        grouped: dict[tuple[str, str], list[dict[str, Any]]],
        num_negatives: int,
        instruction: str,
    ) -> None:
        self.queries = bundle["queries"]
        self.episodes = bundle["episodes"]
        self.episode_by_id = episode_by_id
        self.grouped = grouped
        self.num_negatives = num_negatives
        self.instruction = instruction

    def __len__(self) -> int:
        return len(self.queries)

    def __getitem__(self, idx: int) -> dict[str, Any]:
        query_item = self.queries[idx]
        target_id = query_item["target_episode_ids"][0]
        positive_episode = self.episode_by_id[target_id]
        negatives = sample_hard_negatives(
            query_item=query_item,
            episode_by_id=self.episode_by_id,
            grouped=self.grouped,
            all_episodes=self.episodes,
            num_negatives=self.num_negatives,
        )

        return {
            "query_text": build_query_text(query_item["query"], self.instruction, QUERY_FORMAT),
            "positive_text": build_episode_text(positive_episode),
            "negative_texts": [build_episode_text(ep) for ep in negatives],
            "target_episode_id": target_id,
        }


## Collator

In [ ]:
@dataclass
class EpisodicBatch:
    query_texts: list[str]
    positive_texts: list[str]
    negative_texts: list[list[str]]
    target_episode_ids: list[str]


def collate_fn(items: list[dict[str, Any]]) -> EpisodicBatch:
    return EpisodicBatch(
        query_texts=[item["query_text"] for item in items],
        positive_texts=[item["positive_text"] for item in items],
        negative_texts=[item["negative_texts"] for item in items],
        target_episode_ids=[item["target_episode_id"] for item in items],
    )

## Load model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModel.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=DTYPE if DEVICE == "cuda" else torch.float32,
)

base_model.config.use_cache = False
if hasattr(base_model, "gradient_checkpointing_enable"):
    base_model.gradient_checkpointing_enable()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

## LoRA Adapter

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.FEATURE_EXTRACTION,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
model.to(DEVICE)

trainable params: 10,092,544 || all params: 605,869,056 || trainable%: 1.6658


PeftModelForFeatureExtraction(
  (base_model): LoraModel(
    (model): Qwen3Model(
      (embed_tokens): Embedding(151669, 1024)
      (layers): ModuleList(
        (0-27): 28 x Qwen3DecoderLayer(
          (self_attn): Qwen3Attention(
            (q_proj): lora.Linear(
              (base_layer): Linear(in_features=1024, out_features=2048, bias=False)
              (lora_dropout): ModuleDict(
                (default): Dropout(p=0.05, inplace=False)
              )
              (lora_A): ModuleDict(
                (default): Linear(in_features=1024, out_features=16, bias=False)
              )
              (lora_B): ModuleDict(
                (default): Linear(in_features=16, out_features=2048, bias=False)
              )
              (lora_embedding_A): ParameterDict()
              (lora_embedding_B): ParameterDict()
              (lora_magnitude_vector): ModuleDict()
            )
            (k_proj): lora.Linear(
              (base_layer): Linear(in_features=1024, out_featu

## EOS Pooling Encoder

In [ ]:
class Qwen3Embedder(nn.Module):
    def __init__(self, model: nn.Module, tokenizer) -> None:
        super().__init__()
        self.model = model
        self.tokenizer = tokenizer

    def encode(self, texts: list[str], max_length: int = MAX_LENGTH) -> torch.Tensor:
        eos = self.tokenizer.eos_token or ""
        if eos:
            texts = [text if text.rstrip().endswith(eos) else text + eos for text in texts]

        batch = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        outputs = self.model(**batch)
        hidden = outputs.last_hidden_state
        attention_mask = batch["attention_mask"]
        last_token_indices = attention_mask.sum(dim=1) - 1
        pooled = hidden[torch.arange(hidden.size(0), device=hidden.device), last_token_indices]
        return F.normalize(pooled, p=2, dim=-1)


embedder = Qwen3Embedder(model, tokenizer)

## InfoNCE Loss

In [ ]:
def compute_infonce_loss(
    query_embeds: torch.Tensor,
    positive_embeds: torch.Tensor,
    negative_embeds: torch.Tensor,
    temperature: float = TEMPERATURE,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    query_embeds:    [B, D]
    positive_embeds: [B, D]
    negative_embeds: [B, K, D]
    """
    batch_size = query_embeds.size(0)
    neg_flat = negative_embeds.reshape(-1, negative_embeds.size(-1))

    candidate_docs = torch.cat([positive_embeds, neg_flat], dim=0)  # [B + B*K, D]
    logits = torch.matmul(query_embeds, candidate_docs.t()) / temperature
    labels = torch.arange(batch_size, device=logits.device)
    loss = F.cross_entropy(logits, labels)
    return loss, logits

## Retrieval Evaluation

In [ ]:
@torch.no_grad()
def encode_corpus(texts: list[str], batch_size: int = EVAL_BATCH_SIZE) -> torch.Tensor:
    all_embeddings = []
    for start in range(0, len(texts), batch_size):
        chunk = texts[start : start + batch_size]
        chunk_embeds = embedder.encode(chunk)
        all_embeddings.append(chunk_embeds.cpu())
    return torch.cat(all_embeddings, dim=0)


@torch.no_grad()
def evaluate_bundle(bundle: dict[str, Any]) -> dict[str, float]:
    embedder.model.eval()

    episode_texts = [build_episode_text(ep) for ep in bundle["episodes"]]
    episode_ids = [ep["episode_id"] for ep in bundle["episodes"]]
    episode_embeds = encode_corpus(episode_texts)
    id_to_index = {episode_id: idx for idx, episode_id in enumerate(episode_ids)}

    query_texts = [build_query_text(item["query"], INSTRUCTION, QUERY_FORMAT) for item in bundle["queries"]]
    query_embeds = encode_corpus(query_texts)

    sims = query_embeds @ episode_embeds.t()

    recall_at_1 = 0
    recall_at_5 = 0
    reciprocal_ranks = []

    for i, query_item in enumerate(bundle["queries"]):
        target_id = query_item["target_episode_ids"][0]
        target_index = id_to_index[target_id]

        ranking = torch.argsort(sims[i], descending=True).tolist()
        rank = ranking.index(target_index) + 1

        recall_at_1 += int(rank <= 1)
        recall_at_5 += int(rank <= 5)
        reciprocal_ranks.append(1.0 / rank if rank <= 10 else 0.0)

    n = len(bundle["queries"])
    return {
        "recall@1": recall_at_1 / n,
        "recall@5": recall_at_5 / n,
        "mrr@10": float(np.mean(reciprocal_ranks)),
    }

## DataLoaders

In [ ]:
train_dataset = EpisodicContrastiveDataset(
    bundle=train_bundle,
    episode_by_id=train_episode_by_id,
    grouped=train_grouped,
    num_negatives=NUM_HARD_NEGATIVES,
    instruction=INSTRUCTION,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    drop_last=False,
)

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
total_train_steps = steps_per_epoch * NUM_EPOCHS
warmup_steps = int(total_train_steps * WARMUP_RATIO)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_train_steps,
)

print("steps_per_epoch:", steps_per_epoch)
print("total_train_steps:", total_train_steps)
print("warmup_steps:", warmup_steps)

steps_per_epoch: 7
total_train_steps: 56
warmup_steps: 5


## Training

In [ ]:
best_val_r1 = -1.0
best_metrics = None
global_step = 0

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    optimizer.zero_grad()

    for step, batch in enumerate(train_loader, start=1):
        query_embeds = embedder.encode(batch.query_texts)
        positive_embeds = embedder.encode(batch.positive_texts)

        flat_negatives = [neg for neg_list in batch.negative_texts for neg in neg_list]
        negative_embeds = embedder.encode(flat_negatives)
        negative_embeds = negative_embeds.view(
            len(batch.query_texts),
            NUM_HARD_NEGATIVES,
            -1,
        )

        loss, _ = compute_infonce_loss(
            query_embeds=query_embeds,
            positive_embeds=positive_embeds,
            negative_embeds=negative_embeds,
            temperature=TEMPERATURE,
        )

        loss = loss / GRAD_ACCUM_STEPS
        loss.backward()
        running_loss += loss.item() * GRAD_ACCUM_STEPS

        if step % GRAD_ACCUM_STEPS == 0 or step == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1

    val_metrics = evaluate_bundle(val_bundle)
    epoch_loss = running_loss / len(train_loader)

    print(
        f"epoch={epoch + 1} "
        f"train_loss={epoch_loss:.4f} "
        f"val_r@1={val_metrics['recall@1']:.4f} "
        f"val_r@5={val_metrics['recall@5']:.4f} "
        f"val_mrr@10={val_metrics['mrr@10']:.4f}"
    )

    if val_metrics["recall@1"] > best_val_r1:
        best_val_r1 = val_metrics["recall@1"]
        best_metrics = val_metrics
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)
        print(f"saved best adapter to {OUTPUT_DIR}")

print("best_metrics:", best_metrics)

epoch=1 train_loss=1.1110 val_r@1=1.0000 val_r@5=1.0000 val_mrr@10=1.0000
saved best adapter to qwen3-episodic-lora
epoch=2 train_loss=0.2735 val_r@1=1.0000 val_r@5=1.0000 val_mrr@10=1.0000
epoch=3 train_loss=0.1994 val_r@1=1.0000 val_r@5=1.0000 val_mrr@10=1.0000
epoch=4 train_loss=0.1292 val_r@1=1.0000 val_r@5=1.0000 val_mrr@10=1.0000
epoch=5 train_loss=0.1823 val_r@1=1.0000 val_r@5=1.0000 val_mrr@10=1.0000
epoch=6 train_loss=0.2585 val_r@1=1.0000 val_r@5=1.0000 val_mrr@10=1.0000
epoch=7 train_loss=0.1564 val_r@1=1.0000 val_r@5=1.0000 val_mrr@10=1.0000
epoch=8 train_loss=0.2417 val_r@1=1.0000 val_r@5=1.0000 val_mrr@10=1.0000
best_metrics: {'recall@1': 1.0, 'recall@5': 1.0, 'mrr@10': 1.0}


## Retrieval Demo

In [ ]:
@torch.no_grad()
def retrieve_top_k(bundle: dict[str, Any], query: str, top_k: int = 5) -> list[tuple[str, float]]:
    episode_texts = [build_episode_text(ep) for ep in bundle["episodes"]]
    episode_ids = [ep["episode_id"] for ep in bundle["episodes"]]

    query_embed = encode_corpus([build_query_text(query, INSTRUCTION, QUERY_FORMAT)], batch_size=1)
    episode_embeds = encode_corpus(episode_texts)
    sims = (query_embed @ episode_embeds.t()).squeeze(0)
    top_indices = torch.topk(sims, k=top_k).indices.tolist()

    return [(episode_ids[i], float(sims[i])) for i in top_indices]


sample_query = val_bundle["queries"][0]["query"]
retrieve_top_k(val_bundle, sample_query, top_k=5)

[('ep_archive_low_rated_assets_archive_low_rated_assets_c03_anchor_hawaii_rejects_hawaii_1_winter_2022',
  0.921875),
 ('ep_archive_low_rated_assets_archive_low_rated_assets_c03_near_location_hawaii_rejects_tokyo_1_winter_2022',
  0.71484375),
 ('ep_archive_low_rated_assets_archive_low_rated_assets_c03_near_rating_threshold_hawaii_rejects_hawaii_4_winter_2022',
  0.63671875),
 ('ep_bulk_like_highlights_bulk_like_highlights_c04_near_rating_threshold_hawaii_best_of_false_hawaii_4',
  0.37109375),
 ('ep_bulk_like_highlights_bulk_like_highlights_c04_anchor_hawaii_best_of_false_hawaii_5',
  0.25390625)]

Frozen Test Evaluation

In [ ]:
test_metrics = evaluate_bundle(test_bundle)
print("test_metrics:", test_metrics)

test_metrics: {'recall@1': 0.9642857142857143, 'recall@5': 1.0, 'mrr@10': 0.9821428571428571}


In [ ]:
from peft import PeftModel

base_eval_model = AutoModel.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=DTYPE if DEVICE == "cuda" else torch.float32,
).to(DEVICE)
base_eval_model.eval()

embedder = Qwen3Embedder(base_eval_model, tokenizer)

base_val_metrics = evaluate_bundle(val_bundle)
base_test_metrics = evaluate_bundle(test_bundle)

print("base_val_metrics:", base_val_metrics)
print("base_test_metrics:", base_test_metrics)


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

base_val_metrics: {'recall@1': 0.7, 'recall@5': 1.0, 'mrr@10': 0.8333333333333333}
base_test_metrics: {'recall@1': 0.6607142857142857, 'recall@5': 1.0, 'mrr@10': 0.8041666666666666}


In [ ]:
base_model_for_lora = AutoModel.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=DTYPE if DEVICE == "cuda" else torch.float32,
)

lora_eval_model = PeftModel.from_pretrained(base_model_for_lora, OUTPUT_DIR)
lora_eval_model.to(DEVICE)
lora_eval_model.eval()

embedder = Qwen3Embedder(lora_eval_model, tokenizer)

lora_val_metrics = evaluate_bundle(val_bundle)
lora_test_metrics = evaluate_bundle(test_bundle)

print("lora_val_metrics:", lora_val_metrics)
print("lora_test_metrics:", lora_test_metrics)


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

lora_val_metrics: {'recall@1': 1.0, 'recall@5': 1.0, 'mrr@10': 1.0}
lora_test_metrics: {'recall@1': 0.9821428571428571, 'recall@5': 1.0, 'mrr@10': 0.9910714285714286}


In [ ]:
import shutil

shutil.make_archive("qwen3-episodic-lora", "zip", "qwen3-episodic-lora")

'/content/qwen3-episodic-lora.zip'

In [ ]:
from google.colab import files

files.download("qwen3-episodic-lora.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>